# Introducing Z3

In this notebook we are going to solve the following exercise using the constraint solver `Z3`.

<p>
<center style="color:blue; background-color:#FED8B1; font-family: Philosopher; font-size:20px; padding: 8px">
    I have as many brothers as I have sisters, but my sister has twice as many brothers as she has sisters.
    How many children does my father have?
</center>
</p>

In [1]:
// load the formatting setup and the style sheet, install Z3 if necessary
#load "../setup.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

The first cell loads the file `setup.fsx` and calls the function `installZ3` defined there.  This function downloads the package `Microsoft.Z3` from the Z3 release page on GitHub if it has not been installed before.  The second cell loads the file `z3.fsx`, which makes Z3 available.  The file `z3.fsx` also defines a small layer on top of the .NET interface of Z3 that makes it possible to write constraints in a similar way as in the Python interface of Z3.

In `Z3`, every variable has to be declared.  In this case, all our variables are integer valued.  The function `Int` declares a variable with the given name.
The object `Int "boys"` is the `Z3` representation of a constraint variable.  We assign this object to the F# variable `boys`. 

In [3]:
let boys  = Int "boys"
let girls = Int "girls"

Next, we create a *solver* object.  This is a constraint solver capable of solving constraint satisfaction problems.

In [4]:
let S = newSolver ()

We can add constraints to this solver object via the method `Add`.
We will use two variables:
 - `boys` is the number of my father's sons,
 - `girls` is the number of my father's daughters.
 
Note that the number of my brothers is `boys - 1` as I am not my own brother.  Similarly, the number of my sister's sisters is `girls - 1`.
So the statement that I have as many brothers as sisters, but my sister has twice as many brothers as she has sisters is formalized as follows.  The operator `.=` is defined in `z3.fsx`.  It creates an equation.  The F# operator `=` can not be used for this purpose, since it compares two values and returns a Boolean value.

In [5]:
S.Add(boys - 1 .= girls)
S.Add(2 * (girls - 1) .= boys)

In [6]:
S

[boys - 1 = girls, 2 * (girls - 1) = boys]

The method `Check` examines whether the given set of constraints is satisfiable.
In general, it can return one of the following results:
- `sat`   is returned if the problem is solvable, (`sat` is short for *satisfiable*)
- `unsat` is returned if the problem is unsatisfiable,
- `unknown` is returned if the constraint solver is not powerful enough to solve the given problem.

In the .NET interface of Z3, these values are called `Status.SATISFIABLE`, `Status.UNSATISFIABLE`, and `Status.UNKNOWN`.

In [7]:
if S.Check() = Status.SATISFIABLE then
    printfn "satisfiable"

satisfiable

To extract the solution of the given problem we use the property `Model`.

In [8]:
let Solution = S.Model
Solution

[boys = 4, girls = 3]

In order to extract the number of boys and girls as F# integers, we use the function `intValue` that is defined in `z3.fsx`.

In [9]:
let b = intValue Solution boys
let g = intValue Solution girls

In [10]:
printfn "My father has %d children: %d boys and %d girls." (b + g) b g
printfn "I have %d - 1 = %d brothers and %d sisters." b (b - 1) g
printfn "My sister has %d - 1 = %d sisters and %d = 2 * %d brothers." g (g - 1) b (g - 1)

My father has 

7

 children: 

4

 boys and 

...

 sisters and 

4

 = 2 * 

2

 brothers.